**This code present prompts stratgies for Women Empowerment.**
Since the first approach not adopt, we adopt only the two later approaches that used in feminism movement.

In [ ]:
pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 578.7 kB/s eta 0:00:00


In [ ]:
import openai
import csv
import random

In [ ]:
openai.api_key='OPENAI_API_KEY'


#Second approach: Specific prompt for each stance

In [ ]:
def create_tweets(prompt_, total_tweet):

    prompt = prompt_
    Generated_tweets = []
    #ask ChatGPT create tweets
    response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=[
            {"role": "system", "content": "You are a tweet generator for Arabic language"},
            {"role": "user", "content": prompt}
        ],
    max_tokens=200,
    n=total_tweet,
    temperature=0.7 #sometimes 1
    )

    #add the responses to the list
    for respon in response['choices']:
      tweet = respon['message']['content'].strip()
      Generated_tweets.append(tweet)

    return Generated_tweets

#save the tweets in csv file
def save_tweets_csv(tweets, label):
    with open('womenempoerment.csv', 'w', newline='', encoding='utf-8-sig') as csvfile:
        colum_names = ['Tweet', 'Label']
        writer = csv.DictWriter(csvfile, fieldnames= colum_names)
        writer.writeheader()

        for tweet in tweets:
            writer.writerow({'Tweet': tweet, 'Label': label})

In [ ]:
#prompts for each stance
#for favour satnce
prompt1 = '''
    Task: create an informal tweet that support women empowerment. The tweet may include errors slang, mention and hashtag.
    Type of Text: Short Tweet (Maximum of 30 words).
    The output: should include the tweet only.

    Example: "تمكين المرأة ومشاركتها في القوى العاملة – غسان عدنان العاني "
    Example: "تظل مشكلة المرأة، بالدرجة الأولى، اقتصادية... تمكينها اقتصاديًّا (استقلالها ماديًّا) هو مفتاح تمكينها الفعلي"
    Example: "تمكين المراه هو ليس تقليد منصب فقط بل في ممارسه صلاحياتها في اتخاذ القرار باستقلاليه"
    '''
#for Against
prompt2 = '''
    Task: create an informal tweet that goes against women empowerment. The tweet may include mention and hashtag.
    Type of Text: Short Tweet (Maximum of 30 words).
    The output: should include the tweet only.

    Example: "  الوظايف كلها خذوها البنات اذا ماتدري والناس كلها عارفه لكن تمكين المرأة الله يشغلها ويشغلهم وخلونا يالشباب عاطلين معطلين"
    Example: "  من يوم جاء زمن تمكين المرأة بالحق والباطل والأمور متغيرة للاسوأ"
    Example: " للاسف الاسر تتفكك من الآن بسبب فرض مفاهيم تمكين المرأة التي جعلتها  تتمرد على رسالتها في الحياة."
        '''
#for NONE stance
prompt3 = '''
    Task: create an informal tweet that may not related to women empowerment movements, but use related hashtag.
    Type of Text: Short Tweet (Maximum of 30 words).
    The output: should include the tweet only.

    Example:"  نطالب بوضع غرامة مشددة على رمي السيجارة المشتعلة تحت السيارات   #تمكين_المرأة"
    Example: ""ما في شيء أحلى من القهوة العربية الفاخرة في الصباح، الحياة تحلى "
    Example: "مشاهدة أفلام الرعب وأنا لوحدي في الليل = رهيب"
'''


In [ ]:
# create tweets by choose the stance prompt 1, 2, or 3 and specify number of tweets
total_tweet= 100 #(no more than 128 as it limited number for n parameter)
label='FAVOR'
synthetic_tweets = create_tweets(prompt1, total_tweet)

# save tweets in CSV
save_tweets_csv(synthetic_tweets, label)

# Third approach: Template prompt for each stance


In [ ]:
#function to ask ChatGPT
def create_tweets(sys_prompt, prompt):

    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": prompt},
        ],
        max_tokens=200,
        n=1,
        temperature=0.7,
    )
    return response.choices[0].message.content.strip()

#function to set the hint sentence and then call create_tweets to create the tweets
def set_propmts(num_tweets, templ, hasht, topic, base_P):
    tweets = []
    templates = templ
    hashtags = hasht
    topics = topic

    for _ in range(num_tweets):
        template = random.choice(templates)
        hashtag = random.choice(hashtags)
        topic = random.choice(topics)
        prompt = f"{template} {topic}. {hashtag}"
        base_prompt= base_P
        full_prompt= "Hint: "+ prompt
        tweet = create_tweets(base_prompt, full_prompt)
        tweets.append(tweet)

    return tweets


In [ ]:
#1: prompts for Favor stance consist of base_prompt1 for the system, and the hint sentence lists' to create hint sentence.
#Important: make sure the lists coherent each for each subset of tweets by add comments to some elements to avoid unwanted results.
base_prompt1='''
    You are a tweet generator for Arabic language.
    Task: create an informal tweet that support women empowerment. Inspire your perspective from the hint.
          The hint is consist of "template, Topic. hashtag", use your creativity to generate the tweet without copy the template.
    Type of Text: Short Tweet (Maximum of 30 words) and include the tweet only.

    Example:  "تكمين_المرأة # [تمكين البنات في المجتمع الرقمي والتكنولوجيا] . البنات هن السلاح اللي يغير ويطور.!"
    Answer: "قولوا لبناتكم إنّهن المفتاح للمستقبل المشرق! بيديهن رح يغيّروا العالم بكل اللي فيهن من طاقة ومواهب."

    Example: "مساواة_الجنسين#  .[دور المرأة في المجتمع] , ..نحن في بداية العصر الذهبي وهو تمكين المرأه"
    Answer: "مشروع التمكين بيبتدي بأدوارنا كنساء! كل امرأة لها قوة مذهلة لتغيير المجتمع ورفع شأن الجنس اللطيف."

    Example: "يوم_المرأة_العالمي# .[اعطاء وظائف للنساء] , ...اختصار الكلام: . بلا شك ان النساء "
    Answer: "في يوم المرأة العالمي، دعونا نمنح النساء فرص العمل! بلا كلام كتير: نساءنا جاهزات لكل تحدي وفرصة في ساحة العمل."
'''
templates1 = [
      # "المرأة مش ضعيفة، قوية وقدها تحقق أحلامها في...! ما نبغى حد يقول غير كذا!",
      # "قوة المرأة تكون من تعاوننا وتضامننا..  ",
      # "البنات هن السلاح اللي يغير ويطور.!",
      # "لما المرأة تتمكن، العالم كله يتمكن. !",
      # "تذكري، القوة مو بس القوة الجسدية، هي القوة العقلية !",
      " ...وتتوالى الأخبار المفرحة للمرأة",
      "...اختصار الكلام: . بلا شك ان النساء ",
      # " ...الكل متساوي في موضوع",
      # " ..نحن في بداية العصر الذهبي وهو تمكين المرأه",
      "...أصبح تمكين المرأة واقع نعيشه في",
      # "...ردك يبين تاثرك بالخطاب الرجعي ",
      # "...تمكين المرأة هو حاجة ومطلب رئيسي سيغير في",
      # "شكرًا محمد بن سلمان.. شكرًا قيادتنا الرشيدة  لتمكين المرأة العربية ...",
      # "...تظل مشكلة المرأة، بالدرجة الأولى، اقتصادية",
      "...الشعب الذي يعطي حقوق للمرأة،",
      # "...وين الغلط في التمكين؟",
      # "أطمح إلى تمكين المرأة. و أطمح بأن يكون لي دور في",
      # "...موجة تغييرات في وطنا ابرزها تمكين المرأه",
      "المرأة جزء لا يتجزأ من المجتمع لا يقل مستوى من الرجل"
       ]
hashtags1=[
      "#تكمين_المرأة",
      # "#المرأة_المستقلة ",
      # "#مساواة_الجنسين",
      "#عصر_المرأه",
      "#يوم_المرأة_العالمي"
]
topics1 = [
       # "[مستقبل أكثر تكافؤ وعدل للكل]",
       # "[تمكين المرأة في العمل والقيادة]",
       # "[دور المرأة في المجتمع]",
       # "[تعليم المرأة وتعزيز المعرفة]",
       # "[المرأة في الرياضة والثقافة]",
         "[ رؤية السعودية 2030 للمرأة]",
       # "[ اعطوا النساء حقوقهن]",
         "[اعطاء وظائف للنساء]",
       # "[تمكين البنات في المجتمع الرقمي والتكنولوجيا]",
       # "[دور المرأة وتمكينها في مكان العمل]",
       # "[حقوق المرأة ومكانتها في المجتمع السعودي]",
       # "[مساواة المرأة بالرجل]",
       # "[مساهمات المرأة في التقنية في السعودية]",
       # "[أهمية دعم المرأة في المجتمع]",
       # "[لا يقتصر واجب المرأة على خدمة الرجل]",
         "[حقوق المرأة في التنقل والسفر]",
       # "[تمكين المرأة قتصاديًّا ]",
       # "[القضاء على النظام الأبوي]",
       # "[ تمكين المرأة في كل الميادين]"
    ]

#2: for Aginst stance
base_prompt2='''
    You are a tweet generator for Arabic language.
    Task: create an informal tweet that goes against women empoerment. Inspire your perspective from the hint.
          The hint is consist of "template, Topic. hashtag", use your creativity to generate the tweet without copy the template.
    Type of Text: Short Tweet (Maximum of 30 words) and include the tweet only.

    Example: "تمكين_المرأة# . [نزع الحجاب والتبرج والإنحلال]. تمكين المرأة لايكون على حساب الرجل"
    Answer: "يا حبذا لو يوقفوا عن هالكلام بتمكين المرأة. كله معه نزع حجابها وتبرجها وإنحلال المجتمع!"

    Example:  "المرأة_المستقلة#  [خلونا يالشباب عاطلين]  ,تمكين المرأة لايكون على حساب الرجل"
    Answer:  "شو الفايدة من تمكين المرأة إذا كان على حسابنا كرجال؟ دورنا نوفّر الرعاية والدعم لها، مش نتنافس معاها في المجالات."

    Example: "...تمكين_المرأة# [ واجب المرأة هو خدمة الزوج], المرأة غير مؤهلة للقيادة في "
    Answer: "المرأة لديها دور أكثر أهمية في الحياة: كونها أم وزوجة. لذا دعونا نركز على تعزيز هذه الدور الرئيسي بدلاً من تمكينها في القيادة."

'''
templates2 = [
       #"المرأة تفتقر إلى القدرات والمهارات اللازمة للنجاح في !",
       #"النساء يجب أن يتقيدن بالأدوار التقليدية!",
       #"المرأة غير مؤهلة للقيادة في ... !",
       #"تعليم المرأة لا يعتبر أولوية. هناك أمور أكثر أهمية يجب التركيز عليها!",
       #"...المرأة غير قادرة على مجاراة الرجال !",
        "جاني القرف من شعارات تمكين المرأة...",
       #"تمكين المرأة لايكون على حساب الرجل",
        "...صارت المرأة تتمرد على رسالتها و",
        "تمكين المرأة هو كذبة...",
       #"وش ذا اللي يسمونه تمكين المرأة",
       #"الهدف ليس تمكين المرأة بل استغلالها  ..",
       #" تكمين المرأة غباء",
        "حسب رأيي ليس المقصود من دعم المرأه  الحقوق بل...",
]
hashtags2=[
      "#تكمين_المرأة",
      "#المرأة_المستقلة ",
     #"#مساواة_الجنسين"
]
topics2 = [
    #"[تأثير سيطرة المرأة في العمل]",
    #"[خلونا يالشباب عاطلين]",
    #"[تكمين المرأة يغير القيم والتقاليد]",
     "[الاسر تتفكك بسبب تمكين المرأة]",
     "[نزع الحجاب والتبرج والإنحلال]",
     "[زيادة نسبة الطلاق بسبب تمكين المرأة]",
     "[واجب المرأة الحقيقي هوالأمومة]",
     "[الوظائف هي حق للرجال]",
    #"[ تكمين المرأة يعارض الدين والعادات]",
    #"[  نرفض تقليد الغرب في تمكين المرأة ]",
     "[ واجب المرأة هو خدمة الزوج]" ,
     "[  عزوف الشباب عن الزواج بسبب تمكين المرأة]",
    #"[رؤية_السعودية_2030]",
    #"[تكمين المرأة يتعارض مع التقاليد الإسلامية ]"
]

#none
base_prompt3='''
    You are a tweet generator for Arabic language.
    Task: create an informal tweet that not have clear stance (netural) or not related to women empowerment. Inspire your perspective from the hint. use the exact hashtag.
          The hint is consist of "template, Topic. hashtag", use your creativity to generate the tweet without copy the template.
    Type of Text: Short Tweet (Maximum of 30 words) and include the tweet only.

    Example: "تكمين_المرأة# [علم النفس والاجتماع]  , ...واجب شرعي على الانسان تجاه نفسه أن يتخلص من العلاقات"
    Answer: " علم النفس والاجتماع، مجال مشوّق لفهم البشرية والعلاقات! يمكن اكتشاف أشياء رائعة بدون الحاجة للتخلص من كل العلاقات. #تكمين_المرأة "

    Example: "#يوم_المرأة_العالمي# [كرة القدم]  , ...واجب شرعي على الانسان تجاه نفسه أن يتخلص من العلاقات"
    Answer: "ليش ما نجمع كل الفانز ونشجع فريقنا المفضل؟ كرة القدم بتخلّينا ننسى العلاقات المزعجة ونتجمّع حول هدف واحد #يوم_المرأة_العالمي"

'''
templates3 = [
       # "يجب التركيز على تحقيق توازن الفرص في ... بغض النظر عن الجنس أو النوع",
       # "...التوازن بين العمل والحياة الشخصية يسهم في رفاهية وسعادة الفرد في",
       # "...واجب شرعي على الانسان تجاه نفسه أن يتخلص من العلاقات",
       "نطالب بوضع غرامة مشددة على",
       # "منتدى الفكر العربي يناقش",
       "فاهمه تمكين المرأة غلط "
]
hashtags3=[
    "#تكمين_المرأة",
    #"#عصر_المرأه",
    #"#يوم_المرأة_العالمي",
    "#القبض_علي_مدعيه_النبوه "

]
topics3= [
    #"[التوعية بالصحة العقلية ورفاهية الفرد]",
    #"[تكنولوجيا المستقبل وتطوراتها]",
     "[كرة القدم]",
    #"[هيئة الاعلام والاتصالات ]",
    #"[علم النفس والاجتماع]",
     "[مدعيه النبوه]"

]

In [ ]:
#call the function for each stance separately
num_tweets = 50  #set number of tweets to create subset of tweets by choos the elements of the lists
feminist_favour = set_propmts(num_tweets, templates1, hashtags1, topics1, base_prompt1 )
#feminist_against = set_propmts(num_tweets, templates2, hashtags2, topics2, base_prompt2 )
#feminist_none = set_propmts(num_tweets, templates3, hashtags3, topics3, base_prompt3 )

#save the tweets in CSV file
with open("womenempoerment_template_fav.csv", 'w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerow(["Tweet", "Label"])
    for data in feminist_favour:
        writer.writerow([data, "FAVOR"])